# Deep learning in genomics

This notebook is based on the [jupyter notebook](https://github.com/abidlabs/deep-learning-genomics-primer/blob/master/A_Primer_on_Deep_Learning_in_Genomics_Public.ipynb) from the publication ["A primer on deep learning in genomics"](https://www.nature.com/articles/s41588-018-0295-5) but uses the [fastai](https://www.fast.ai) library based on [PyTorch](https://pytorch.org).

# Notebook setup

In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from fastai import *
from fastai.vision import *

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

In [ ]:
# fastai version
__version__

# Data setup

## Load data from the web, generate dataframe, and save to disk

In [ ]:
import requests

In [ ]:
URL_seq = 'https://raw.githubusercontent.com/abidlabs/deep-learning-genomics-primer/master/sequences.txt'

In [ ]:
# get data from URL
seq_raw = requests.get(URL_seq).text.split('\n')
seq_raw = list(filter(None, seq_raw)) # Removes empty lists

In [ ]:
# check length
len(seq_raw)

In [ ]:
# setup df from list
seq_df = pd.DataFrame(seq_raw, columns=['Sequences'])

In [ ]:
# show head of dataframe
#seq_df.head()

In [ ]:
URL_labels = 'https://raw.githubusercontent.com/abidlabs/deep-learning-genomics-primer/master/labels.txt'

In [ ]:
seq_labels = requests.get(URL_labels).text.split('\n')
seq_labels = list(filter(None, seq_labels)) # Removes empty entries

In [ ]:
len(seq_labels)

In [ ]:
seq_label_series = pd.Series(seq_labels)

In [ ]:
seq_df['Target'] = seq_label_series.astype('int')

In [ ]:
seq_df.head()

In [ ]:
seq_df.to_csv('seq_df.csv')

## Data loading

In [ ]:
seq_df = pd.read_csv('seq_df.csv', index_col=0)

In [ ]:
#seq_df.drop('Unnamed: 0', axis=1, inplace=True)

In [ ]:
seq_df.head()

In [ ]:
# Preprocess y values

In [ ]:
len(seq_df['Target'])

In [ ]:
targA = seq_df['Target'].values; targA

In [ ]:
targB = np.logical_not(seq_df['Target'].values).astype(int); targB

In [ ]:
targ = [[a,b] for a, b in zip(targA, targB)]; targ[:4], '...', targ[-4:]#, len(targ)

In [ ]:
seq_df['MultiTarget'] = targ

In [ ]:
def joinarray(x):
    y = ';'.join((str(x[0]),str(x[1])))
    return y

In [ ]:
seq_df['MultiTarget'] = seq_df['MultiTarget'].apply(joinarray)

In [ ]:
seq_df.head()

## Data encoding test (incorporated into "open_seq_image" function)

In [ ]:
# setup class instance to encode the four different bases to integer values (1D)
int_enc = LabelEncoder()

In [ ]:
# setup one hot encoder to encode integer encoded classes (1D) to one hot encoded array (4D)
one_hot_enc = OneHotEncoder(categories=[range(4)])

In [ ]:
seq_enc = []

for s in seq:
    enc = int_enc.fit_transform(list(s)) # bases (ACGT) to int (0,1,2,3)
    enc = np.array(enc).reshape(-1,1) # reshape to get rank 2 array (from rank 1 array)
    enc = one_hot_enc.fit_transform(enc) # encoded integer encoded bases to sparse matrix (sparse matrix dtype)
    seq_enc.append(enc.toarray()) # export sparse matrix to np array

In [ ]:
len(seq_enc)

In [ ]:
seq_enc[0].T, seq_enc[0].T.shape

In [ ]:
PIL.Image.fromarray(seq_enc[0].T.astype('uint8')*255).convert('P')

## fastai data setup for NN

In [ ]:
# open sequence image function
def open_seq_image(seq:str, cls:type=Image)->Image:
    "Return `Image` object created from sequence string `seq`."
    
    int_enc = LabelEncoder() # setup class instance to encode the four different bases to integer values (1D)
    one_hot_enc = OneHotEncoder(categories=[range(4)]) # setup one hot encoder to encode integer encoded classes (1D) to one hot encoded array (4D)
    
    enc = int_enc.fit_transform(list(seq)) # bases (ACGT) to int (0,1,2,3)
    enc = np.array(enc).reshape(-1,1) # reshape to get rank 2 array (from rank 1 array)
    enc = one_hot_enc.fit_transform(enc) # encoded integer encoded bases to sparse matrix (sparse matrix dtype)
    enc = enc.toarray().T # export sparse matrix to np array
    #print('enc', enc, enc.shape)
    
    # https://stackoverflow.com/questions/22902040/convert-black-and-white-array-into-an-image-in-python
    x = PIL.Image.fromarray(enc.astype('uint8')).convert('P')
    x = pil2tensor(x,np.float32)
    #x = x.view(4,-1) # remove first dimension
    #x = x.expand(3, 4, 50) # expand to 3 channel image
    #print('x', x, x.shape)
    return cls(x)

In [ ]:
# test open sequence image function
open_seq_image('CCGAGGGCTATGGTTTGGAAGTTAGAACCCTGGGGCTTCTCGCGGACACC')

In [ ]:
class SeqItemList(ImageItemList):
    _bunch,_square_show = ImageDataBunch,True
    def __post_init__(self):
        super().__post_init__()
        self.sizes={}
    
    def open(self, seq): return open_seq_image(seq)
    
    def get(self, i):
        seq = self.items[i][0]
        res = self.open(seq)
        return res
    
    @classmethod
    def import_from_df(cls, df:DataFrame, cols:IntsOrStrs=0, **kwargs)->'ItemList':
        "Get the sequences in `col` of `df` and will had `path/folder` in front of them, `suffix` at the end."
        return cls(items=df[cols].values)

In [ ]:
bs = 64

In [ ]:
data = (SeqItemList.import_from_df(seq_df, ['Sequences'])
        .random_split_by_pct(valid_pct=0.25)
        #.split_by_idxs(range(1500), range(1500,2000))
        #.label_from_list(seq_df['Target'].values) # --> Two categories! AND WRONG LOSS FUNCTION!?
        #.label_from_list(targ) # --> MultiCategory
        #.label_from_list(seq_df['MultiTarget'], sep=';')
        .label_from_list(seq_df['Target'].values, sep=';')
        .databunch(bs=bs))

## Verify data setup

### Check data object

In [ ]:
next(iter(data.train_dl))

In [ ]:
data

In [ ]:
# check classes
data.c, data.classes

In [ ]:
data.train_dl.batch_size

### Check data points

In [ ]:
i = 2
data.x[i]

In [ ]:
data.y[i]

In [ ]:
i = 3
data.x[i]

In [ ]:
data.y[i]

In [ ]:
data.show_batch(rows=2)

# Model setup

## Setup fastai ResNet34 architecture

In [ ]:
test_learn = create_cnn(data, models.resnet18)

In [ ]:
test_learn.loss_func.func

In [ ]:
#test_learn.model[0]

In [ ]:
test_learn.model[1]

In [ ]:
def ExpandInput(): return Lambda(lambda x: x.expand(-1, 3, 4, 50))

In [ ]:
EI = ExpandInput()

In [ ]:
# Test ExpandInput layer

In [ ]:
tt = torch.rand((64,1,4,50)); tt.shape

In [ ]:
tt.expand(-1, 3, 4, 50).shape

In [ ]:
Image(EI(tt)[0])

In [ ]:
new_model = nn.Sequential(ExpandInput(), test_learn.model) # insert ExpandInput layer at the beginning of the network

In [ ]:
new_model[0](tt).shape

In [ ]:
new_learn = Learner(data, new_model)#, metrics=accuracy)

In [ ]:
#[p.shape for p in new_model.parameters()]

In [ ]:
new_learn.fit_one_cycle(5)

In [ ]:
new_learn.recorder.plot_losses()

In [ ]:
test_learn.get_predicts()

In [ ]:
del test_learn
del new_model
del new_learn

## Setup custom model

In [ ]:
def Flatten(): return Lambda(lambda x: x.view((x.size(0), -1)))

In [ ]:
def ResizeInput(): return Lambda(lambda x: x.view((-1,)+x.size()[-2:]))

In [ ]:
#def ResizeOutput(): return Lambda(lambda x: x.view(-1))

In [ ]:
drop_p = 0.2

In [ ]:
# inplace=True seems to generate problems?

In [ ]:
net = nn.Sequential(ResizeInput(),
                    nn.Conv1d(in_channels=4, out_channels=32, kernel_size=12),
                    nn.MaxPool1d(kernel_size=4),
                    Flatten(),
                    nn.Dropout(drop_p),
                    nn.Linear(in_features=288, out_features=16),
                    nn.ReLU(),
                    nn.Dropout(drop_p),
                    nn.Linear(in_features=16, out_features=2),
                    #nn.Dropout(drop_p), # not at the end?
                    #Debugger()
                   )

In [ ]:
net

In [ ]:
apply_init(net, nn.init.kaiming_normal_)

## Run sequence through network for testing

In [ ]:
data.train_ds[0], data.train_ds[3]

In [ ]:
net(data.train_ds[0][0].data), net(data.train_ds[3][0].data)

# Learner setup

## TBLogger

In [ ]:
import tensorflow as tf

# From https://gist.github.com/gyglim/1f8dfb1b5c82627ae3efcfbbadb9f514
"""Simple example on how to log scalars and images to tensorboard without tensor ops.
License: Copyleft
"""
#__author__ = "Michael Gygli"

#import tensorflow as tf
#from StringIO import StringIO
#import matplotlib.pyplot as plt
#import numpy as np

class Logger(object):
    """Logging in tensorboard without tensorflow ops."""

    def __init__(self, log_dir):
        """Creates a summary writer logging to log_dir."""
        self.writer = tf.summary.FileWriter(log_dir)

    def log_scalar(self, tag, value, step):
        """Log a scalar variable.
        Parameter
        ----------
        tag : basestring
            Name of the scalar
        value
        step : int
            training iteration
        """
        summary = tf.Summary(value=[tf.Summary.Value(tag=tag,
                                                     simple_value=value)])
        self.writer.add_summary(summary, step)

    def log_images(self, tag, images, step):
        """Logs a list of images."""

        im_summaries = []
        for nr, img in enumerate(images):
            # Write the image to a string
            s = StringIO()
            plt.imsave(s, img, format='png')

            # Create an Image object
            img_sum = tf.Summary.Image(encoded_image_string=s.getvalue(),
                                       height=img.shape[0],
                                       width=img.shape[1])
            # Create a Summary value
            im_summaries.append(tf.Summary.Value(tag='%s/%d' % (tag, nr),
                                                 image=img_sum))

        # Create and write Summary
        summary = tf.Summary(value=im_summaries)
        self.writer.add_summary(summary, step)
        

    def log_histogram(self, tag, values, step, bins=1000):
        """Logs the histogram of a list/vector of values."""
        # Convert to a numpy array
        values = np.array(values)
        
        # Create histogram using numpy        
        counts, bin_edges = np.histogram(values, bins=bins)

        # Fill fields of histogram proto
        hist = tf.HistogramProto()
        hist.min = float(np.min(values))
        hist.max = float(np.max(values))
        hist.num = int(np.prod(values.shape))
        hist.sum = float(np.sum(values))
        hist.sum_squares = float(np.sum(values**2))

        # Requires equal number as bins, where the first goes from -DBL_MAX to bin_edges[1]
        # See https://github.com/tensorflow/tensorflow/blob/master/tensorflow/core/framework/summary.proto#L30
        # Thus, we drop the start of the first bin
        bin_edges = bin_edges[1:]

        # Add bin edges and counts
        for edge in bin_edges:
            hist.bucket_limit.append(edge)
        for c in counts:
            hist.bucket.append(c)

        # Create and write Summary
        summary = tf.Summary(value=[tf.Summary.Value(tag=tag, histo=hist)])
        self.writer.add_summary(summary, step)
        self.writer.flush()
        
"A `Callback` that saves tracked metrics into a log file for Tensorboard."
# Based on https://gist.github.com/gyglim/1f8dfb1b5c82627ae3efcfbbadb9f514
# and devforfu: https://nbviewer.jupyter.org/gist/devforfu/ea0b3fcfe194dad323c3762492b05cae
# Contribution from MicPie

#from ..torch_core import *
#from ..basic_data import DataBunch
#from ..callback import *
#from ..basic_train import Learner, LearnerCallback
#import tensorflow as tf

__all__ = ['TBLogger']

@dataclass
class TBLogger(LearnerCallback):
    "A `LearnerCallback` that saves history of metrics while training `learn` into log files for Tensorboard."
    
    log_dir:str = 'logs'
    log_name:str = 'data'
    log_scalar:bool = True # log scalar values for Tensorboard scalar summary
    log_hist:bool = True # log values and gradients of the parameters for Tensorboard histogram summary
    log_img:bool = False # log values for Tensorboard image summary

    def __post_init__(self): 
        super().__post_init__()
    #def __init__(self):
    #    super().__init__()
        self.path = self.learn.path
        (self.path/self.log_dir).mkdir(parents=True, exist_ok=True) # setup logs directory
        self.Log = Logger(str(self.path/self.log_dir/self.log_name))
        self.epoch = 0
        self.batch = 0
        self.log_grads = {}
    
    def on_backward_end(self, **kwargs:Any):
        self.batch = self.batch+1
        #print('\nBatch: ',self.batch)
        
        if self.log_hist:
            for tag, value in learn.model.named_parameters():
                tag_grad = tag.replace('.', '/')+'/grad'
                
                if tag_grad in self.log_grads:
                    #self.log_grads[tag_grad] += value.grad.data.cpu().detach().numpy()
                    self.log_grads[tag_grad] = self.log_grads[tag_grad] + value.grad.data.cpu().detach().numpy() # gradients are summed up from every batch
                    #print('if')
                else:
                    self.log_grads[tag_grad] = value.grad.data.cpu().detach().numpy()
                    #print('else')
                
                #print(tag_grad, self.log_grads[tag_grad].sum())
        return self.log_grads
    
    #def on_step_end(self, **kwards:Any):
        #print('Step end: ', self.log_grads)

    def on_epoch_end(self, epoch:int, smooth_loss:Tensor, last_metrics:MetricsList, **kwargs:Any) -> bool:
        last_metrics = ifnone(last_metrics, [])
        tr_info = {name: stat for name, stat in zip(self.learn.recorder.names, [epoch, smooth_loss] + last_metrics)}
        self.epoch = tr_info['epoch']
        self.batch = 0 # reset batch count
        #print('\nEpoch: ',self.epoch)
        
        if self.log_scalar:
            for tag, value in tr_info.items():
                if tag == 'epoch': continue
                self.Log.log_scalar(tag, value, self.epoch+1)
                
        if self.log_hist:
            for tag, value in learn.model.named_parameters():
                
                tag = tag.replace('.', '/')
                self.Log.log_histogram(tag, value.data.cpu().numpy(), self.epoch+1)
                
                tag_grad = tag.replace('.', '/')+'/grad'
                self.Log.log_histogram(tag_grad, self.log_grads[tag_grad], self.epoch+1)
                #print(tag_grad, self.log_grads[tag_grad].sum())
                
        #if self.log_img:
        #    for tag, value in learn.model.named_parameters():
        #        
        #        tag = tag.replace('.', '/')
        #        self.Log.log_images(tag, value.data.cpu().numpy(), self.epoch+1)

## fastai learner setup

In [ ]:
#learn = Learner(data, net, loss_func=nn.functional.cross_entropy, metrics=accuracy)#, callback_fns=[TBLogger])

In [ ]:
def accuracy_float(input:Tensor, targs:LongTensor)->Rank0Tensor:
    "Compute accuracy with `targs` when `input` is bs * n_classes."
    n = targs.shape[0]
    input = input.argmax(dim=-1).view(n,-1)
    targs = targs.type(torch.long) # convert to torch.long = int64 --> difference to accuracy metric!
    targs = targs.view(n,-1)
    return (input==targs).float().mean()

In [ ]:
learn = Learner(data, net, loss_func=BCEWithLogitsFlat(), metrics=accuracy_float)#, callback_fns=[TBLogger])

In [ ]:
learn.summary()

## fastai training

In [ ]:
learn.lr_find()
learn.recorder.plot()

In [ ]:
learn.fit_one_cycle(10, max_lr=1e-2)

In [ ]:
learn.recorder.plot_losses()

In [ ]:
learn.recorder.plot_metrics()

In [ ]:
learn.get_preds(DatasetType.Train)

In [ ]:
data.train_ds[0][1], data.train_ds[3][1]

In [ ]:
#net(data.train_ds[0][0].data), net(data.train_ds[3][0].data)

In [ ]:
net(data.train_ds[0][0].data), net(data.train_ds[3][0].data)

In [ ]:
preds, targs = learn.get_preds()

In [ ]:
#targs.view(-1,1)

In [ ]:
#preds

In [ ]:
i = 0
preds[i], targs[i]

In [ ]:
data.train_ds[0], data.train_ds[3]

In [ ]:
learn.predict(data.train_ds[0][0]), learn.predict(data.train_ds[3][0])

In [ ]:
#learn.save('v6')

In [ ]:
#learn.load('v6')

In [ ]:
interpret = ClassificationInterpretation.from_learner(learn)

In [ ]:
interpret.confusion_matrix()

In [ ]:
#interpret.plot_confusion_matrix()

## Basic PyTorch

In [ ]:
def seq2array(seq:str)->List:
    "Return `List` object with np.array created from sequence string `seq`."
    
    int_enc = LabelEncoder() # setup class instance to encode the four different bases to integer values (1D)
    one_hot_enc = OneHotEncoder(categories=[range(4)]) # setup one hot encoder to encode integer encoded classes (1D) to one hot encoded array (4D)
    
    enc = int_enc.fit_transform(list(seq)) # bases (ACGT) to int (0,1,2,3)
    enc = np.array(enc).reshape(-1,1) # reshape to get rank 2 array (from rank 1 array)
    enc = one_hot_enc.fit_transform(enc) # encoded integer encoded bases to sparse matrix (sparse matrix dtype)
    enc = enc.toarray().T # export sparse matrix to np array
    
    return enc

In [ ]:
# test open sequence image function
test_arr = seq2array('CCGAGGGCTATGGTTTGGAAGTTAGAACCCTGGGGCTTCTCGCGGACACC'); test_arr, test_arr.shape

In [ ]:
plt.imshow(test_arr)
plt.axis('off');

In [ ]:
test_ten = tensor(test_arr).view(1,4,50).type(torch.float); test_ten

In [ ]:
net(test_ten)

In [ ]:
#x = [seq2array(s) for s in seq_df['Sequences'].values]
x = [tensor(seq2array(s)).view(1,4,50).type(torch.float) for s in seq_df['Sequences'].values]

In [ ]:
x = torch.stack(x) # convert list to tensor

In [ ]:
y = tensor(targ).type(torch.float)

In [ ]:
len(x), len(y)

In [ ]:
i = 2
x[i], y[i]

In [ ]:
#train_ds = TensorDataset(*map(tensor, (x[:1500],y[:1500])))
#valid_ds = TensorDataset(*map(tensor, (x[-500:],y[-500:])))

In [ ]:
train_ds = TensorDataset(x[:1500],y[:1500])
valid_ds = TensorDataset(x[-500:],y[-500:])

In [ ]:
len(train_ds), len(valid_ds)

In [ ]:
bs = 64

In [ ]:
train_dl = DataLoader(train_ds, batch_size=bs, shuffle=True)
valid_dl = DataLoader(valid_ds, batch_size=bs*2, shuffle=False)

In [ ]:
opt = optim.SGD(net.parameters(), lr=1e-3, momentum=0.9)

In [ ]:
# From https://github.com/fastai/fastai_docs/blob/master/dev_nb/mnist_sample.py

def simple_loss_batch(model, loss_func, xb, yb, opt=None):
    loss = loss_func(model(xb), yb)

    if opt is not None:
        loss.backward()
        opt.step()
        opt.zero_grad()

    return loss.item(), len(xb)

def simple_fit(epochs, model, loss_func, opt, train_dl, valid_dl):
    for epoch in range(epochs):
        model.train()
        for xb,yb in train_dl: simple_loss_batch(model, loss_func, xb, yb, opt)

        model.eval()
        with torch.no_grad():
            losses,nums = zip(*[simple_loss_batch(model, loss_func, xb, yb)
                                for xb,yb in valid_dl])
            
            # Accuracy metric:
            thr = 0.5
            comparison = [np.array(torch.sigmoid(net(xb)).detach().numpy()>thr).astype(int)==yb.detach().numpy().astype(int)
                          for xb,yb in valid_dl]
            accuracy = np.concatenate(comparison).mean()
            
        val_loss = np.sum(np.multiply(losses,nums)) / np.sum(nums)

        #if epoch % 10 == 0:
        print(epoch, val_loss, accuracy)

In [ ]:
simple_fit(50, net, F.binary_cross_entropy_with_logits, opt, train_dl, valid_dl)

In [ ]:
valid_dl = DataLoader(valid_ds, batch_size=bs*2, shuffle=False)

In [ ]:
xbatch, ybatch = next(iter(valid_dl)); xbatch.shape, ybatch.shape

In [ ]:
ybatch[:10].detach().numpy().astype(int)

In [ ]:
np.array(torch.sigmoid(net(xbatch[:10])).detach().numpy()>0.5).astype(int)

In [ ]:
#np.array(torch.sigmoid(net(xbatch)).detach().numpy()>0.5).astype(int)==ybatch.detach().numpy().astype(int)

In [ ]:
np.array(np.array(torch.sigmoid(net(xbatch)).detach().numpy()>0.5).astype(int)==ybatch.detach().numpy().astype(int)).mean()

In [ ]:
np.concatenate(np.array(torch.sigmoid(net(xbatch)).detach().numpy()>0.5).astype(int)==ybatch.detach().numpy().astype(int)).mean()

In [ ]:
np.concatenate([np.array(torch.sigmoid(net(xb)).detach().numpy()>0.5).astype(int)==yb.detach().numpy().astype(int) for xb,yb in valid_dl]).mean()


In [ ]:
[np.concatenate(np.array(torch.sigmoid(net(xb)).detach().numpy()>0.5).astype(int)==yb.detach().numpy().astype(int)).mean() for xb,yb in valid_dl]


In [ ]:
data.train_ds[0][1], data.train_ds[3][1]

In [ ]:
net(data.train_ds[0][0].data), net(data.train_ds[3][0].data)

# Check fastai dataloader

In [ ]:
data.train_ds.y[0]

In [ ]:
next(iter(data.train_dl))